# Benchmark Comparison: ωe and RPFR

This notebook:
1. Compares harmonic frequencies (ωe) across computational methods vs experiment
2. Computes RPFRs using each method's ωe (with experimental ωexe, B0, αe)
3. Compares the resulting RPFRs against Amila's full computation

Data from `Check_Benchmark.xlsx` (ωe) and `RPFR-CCSD(T)_DTQ (1).xlsx` (Amila RPFR + exp constants).

In [8]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

from richet_rpfr import (
    load_benchmark_we,
    build_benchmark_parents,
    load_experimental_constants,
    load_amila_rpfr,
    generate_all_variants,
    compute_experimental_rpfr,
    compute_benchmark_rpfr,
    compare_amila_to_experimental,
    AMILA_CONTRIB_COLS,
)
from richet_rpfr.isotopes import parse_isotopologue_label, isotopologue_to_formula

pd.set_option("display.float_format", "{:.6f}".format)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Load all data

In [9]:
DATA_FILE = project_root / "RPFR-CCSD(T)_DTQ (1).xlsx"
BENCH_FILE = project_root / "Check_Benchmark.xlsx"

# Amila's computed RPFR values
amila_df = load_amila_rpfr(DATA_FILE)

# Experimental spectroscopic constants
exp_parents = load_experimental_constants(DATA_FILE)

# Benchmark harmonic frequencies
bench_we = load_benchmark_we(BENCH_FILE)

print(f"Amila pairs: {len(amila_df)}")
print(f"Experimental parents: {len(exp_parents)}")
print(f"Benchmark molecules: {len(bench_we)}")
print(f"Benchmark molecules: {list(bench_we.keys())}")

Amila pairs: 131
Experimental parents: 30
Benchmark molecules: 11
Benchmark molecules: ['12C16O', '12C32S', '35Cl35Cl', '1H35Cl', '1H19F', '1H1H', '14N16O', '14N14N', '16O16O', '32S16O', '32S32S']


## 2. Benchmark ωe values

In [10]:
# Reshape benchmark ωe into a table
bench_rows = []
for label, methods in bench_we.items():
    row = {"Isotopologue": label}
    row.update(methods)
    bench_rows.append(row)

bench_table = pd.DataFrame(bench_rows).set_index("Isotopologue")
bench_table

,Experiment,"Amila TQ5 (Molpro, 7pt fit)","Piotr TQ5 (Molpro, 7pt fit)","Piotr TQ5 (Molpro, full path)","Piotr 7Z (Gaussian, full path)","Piotr F12 (Molpro/Orca, full path)",Simon (CCSD(T)/aug-cc-pv5Z)
Isotopologue,,,,,,,
12C16O,2169.755890,2175.460000,2164.790000,2164.693112,2165.385596,2166.459174,2162.540000
12C32S,1285.154640,1292.430000,1284.870000,1284.816938,1285.329227,1285.663945,NaN
35Cl35Cl,559.751000,564.500000,562.080000,562.134887,563.317108,563.304692,NaN
1H35Cl,2990.924800,2997.470000,2991.290000,2991.220435,2995.209010,2994.967300,NaN
1H19F,4138.385000,4146.080000,4140.360000,4140.309311,4142.197401,4140.964510,4141.050000
1H1H,4401.213000,4405.640000,4404.700000,4403.883568,4402.470593,4404.188835,NaN
14N16O,1904.134600,1919.560000,1891.550000,1899.267549,1912.477726,1896.941057,NaN
14N14N,2358.570000,2372.440000,2361.350000,2361.308440,2361.747548,2362.760688,NaN
16O16O,1580.190000,1606.940000,1599.120000,1599.081454,1602.275361,1595.997602,NaN


## 3. Build benchmark parents and compute RPFRs

For each benchmark method, we replace ωe in the experimental parent constants
and keep ωexe, B0, αe from experiment. Then we generate isotopic variants
and compute RPFRs for all Amila pairs.

In [11]:
bench_parents = build_benchmark_parents(bench_we, exp_parents)

print("Benchmark methods and parent counts:")
for method, parents in bench_parents.items():
    print(f"  {method}: {len(parents)} parents")

bench_rpfr = compute_benchmark_rpfr(amila_df, bench_parents, temperature=273.15)

valid = bench_rpfr.dropna(subset=["RPFR 273.15K"])
print(f"\nTotal benchmark RPFR rows: {len(bench_rpfr)}")
print(f"Valid (non-NaN): {len(valid)}")
print(f"Unique isotopologue pairs covered: {valid['Molecule'].nunique()}")

Benchmark methods and parent counts:
  Amila TQ5 (Molpro, 7pt fit): 11 parents
  Piotr 7Z (Gaussian, full path): 11 parents
  Piotr F12 (Molpro/Orca, full path): 11 parents
  Piotr TQ5 (Molpro, 7pt fit): 11 parents
  Piotr TQ5 (Molpro, full path): 11 parents
  Simon (CCSD(T)/aug-cc-pv5Z): 11 parents

Total benchmark RPFR rows: 786
Valid (non-NaN): 291
Unique isotopologue pairs covered: 60


## 4. Also compute experimental RPFR for comparison

In [12]:
exp_lookup = generate_all_variants(exp_parents)
exp_rpfr_df = compute_experimental_rpfr(amila_df, exp_lookup, temperature=273.15)
print(f"Experimental RPFR computed for {len(exp_rpfr_df)} pairs")

Experimental RPFR computed for 131 pairs


## 5. Side-by-side RPFR comparison: Amila vs Benchmark methods

For each isotopologue pair, compare Amila's RPFR against the RPFR computed
from each benchmark method's ωe.

In [13]:
# Pivot benchmark RPFR to wide format
bench_wide = bench_rpfr.pivot_table(
    index="Molecule",
    columns="Benchmark_Method",
    values="RPFR 273.15K",
)

# Merge with Amila and Experimental
rpfr_compare = amila_df[["Molecule", "Method", "RPFR 273.15K"]].copy()
rpfr_compare = rpfr_compare.rename(columns={"RPFR 273.15K": "Amila"})

rpfr_compare = rpfr_compare.merge(
    exp_rpfr_df[["Molecule", "RPFR 273.15K"]].rename(columns={"RPFR 273.15K": "Experimental"}),
    on="Molecule",
    how="left",
)

rpfr_compare = rpfr_compare.merge(bench_wide, on="Molecule", how="left")

# Only show rows with at least one benchmark value
bench_method_cols = [c for c in bench_wide.columns]
has_bench = rpfr_compare[bench_method_cols].notna().any(axis=1)
rpfr_compare = rpfr_compare[has_bench].reset_index(drop=True)

print(f"{len(rpfr_compare)} pairs with benchmark RPFR")
rpfr_compare

63 pairs with benchmark RPFR


,Molecule,Method,Amila,Experimental,"Amila TQ5 (Molpro, 7pt fit)","Piotr 7Z (Gaussian, full path)","Piotr F12 (Molpro/Orca, full path)","Piotr TQ5 (Molpro, 7pt fit)","Piotr TQ5 (Molpro, full path)",Simon (CCSD(T)/aug-cc-pv5Z)
0,1H2H/1H1H,CCSD(T)_Q5,21.559880,21.495294,21.644689,21.620517,21.633618,21.637517,21.631290,NaN
1,1H3H/1H1H,CCSD(T)_Q5,64.551280,64.122800,64.889711,64.790529,64.844280,64.860279,64.834727,NaN
2,2H19F/1H19F,CCSD(T)_Q5,38.205310,38.064531,38.356600,38.248873,38.214728,38.197997,38.196595,38.217095
3,3H19F/1H19F,CCSD(T)_Q5,208.323600,206.839457,209.414327,208.573138,208.306733,208.176233,208.165294,208.325194
4,2H35Cl/1H35Cl,CCSD(T)_Q5,18.078870,17.935379,18.101268,18.070813,18.067561,18.018146,18.017213,NaN
5,3H35Cl/1H35Cl,CCSD(T)_Q5,71.533400,71.151644,71.649351,71.476364,71.457896,71.177512,71.172219,NaN
6,1H37Cl/1H35Cl,CCSD(T)_Q5,1.092419,1.092403,1.092417,1.092412,1.092412,1.092404,1.092404,NaN
7,2H37Cl/1H35Cl,CCSD(T)_Q5,19.782380,19.710360,19.806662,19.773214,19.769641,19.715370,19.714345,NaN
8,3H37Cl/1H35Cl,CCSD(T)_Q5,78.354920,77.935826,78.482705,78.292626,78.272333,77.964249,77.958433,NaN
9,35Cl37Cl/35Cl35Cl,CCSD(T)_Q5,2.192593,2.192284,2.192572,2.192501,2.192500,2.192426,2.192430,NaN


## 6. Per-mil differences: each benchmark method vs Amila

In [14]:
diff_df = rpfr_compare[["Molecule", "Method", "Amila"]].copy()

# Diff vs Experimental
for col in ["Experimental"] + bench_method_cols:
    diff_df[f"{col} (‰)"] = (
        (rpfr_compare[col] - rpfr_compare["Experimental"]) / rpfr_compare["Experimental"]
    ) * 1000.0

diff_df = diff_df.drop(columns=["Experimental (‰)"])
diff_df

,Molecule,Method,Amila,"Amila TQ5 (Molpro, 7pt fit) (‰)","Piotr 7Z (Gaussian, full path) (‰)","Piotr F12 (Molpro/Orca, full path) (‰)","Piotr TQ5 (Molpro, 7pt fit) (‰)","Piotr TQ5 (Molpro, full path) (‰)",Simon (CCSD(T)/aug-cc-pv5Z) (‰)
0,1H2H/1H1H,CCSD(T)_Q5,21.559880,6.950123,5.825600,6.435086,6.616475,6.326776,NaN
1,1H3H/1H1H,CCSD(T)_Q5,64.551280,11.960042,10.413291,11.251543,11.501051,11.102566,NaN
2,2H19F/1H19F,CCSD(T)_Q5,38.205310,7.672987,4.842866,3.945845,3.506312,3.469465,4.008020
3,3H19F/1H19F,CCSD(T)_Q5,208.323600,12.448641,8.381774,7.093790,6.462869,6.409983,7.183048
4,2H35Cl/1H35Cl,CCSD(T)_Q5,18.078870,9.249288,7.551243,7.369883,4.614760,4.562713,NaN
5,3H35Cl/1H35Cl,CCSD(T)_Q5,71.533400,6.995018,4.563775,4.304212,0.363566,0.289169,NaN
6,1H37Cl/1H35Cl,CCSD(T)_Q5,1.092419,0.012999,0.008491,0.008009,0.000678,0.000539,NaN
7,2H37Cl/1H35Cl,CCSD(T)_Q5,19.782380,4.885874,3.188864,3.007615,0.254197,0.202182,NaN
8,3H37Cl/1H35Cl,CCSD(T)_Q5,78.354920,7.017044,4.578128,4.317746,0.364707,0.290076,NaN
9,35Cl37Cl/35Cl35Cl,CCSD(T)_Q5,2.192593,0.131291,0.098740,0.098399,0.064740,0.066247,NaN


## 7. Summary: mean and max |RPFR difference| per method

How different is each benchmark method's RPFR from Amila's, on average?

In [15]:
diff_cols = [c for c in diff_df.columns if c.endswith("(‰)")]

method_summary = pd.DataFrame({
    "Mean (‰)": diff_df[diff_cols].mean(),
    "Mean |‰|": diff_df[diff_cols].abs().mean(),
    "Max |‰|": diff_df[diff_cols].abs().max(),
    "Std (‰)": diff_df[diff_cols].std(),
    "N pairs": diff_df[diff_cols].count(),
})
method_summary.index = [c.replace(" (‰)", "") for c in method_summary.index]
method_summary.index.name = "Method"
method_summary

,Mean (‰),Mean |‰|,Max |‰|,Std (‰),N pairs
Method,,,,,
"Amila TQ5 (Molpro, 7pt fit)",1.873609,1.873609,12.448641,2.750660,63
"Piotr 7Z (Gaussian, full path)",1.279017,1.424140,10.413291,2.475883,49
"Piotr F12 (Molpro/Orca, full path)",1.051115,1.218343,11.251543,2.188374,63
"Piotr TQ5 (Molpro, 7pt fit)",0.786588,1.064194,11.501051,2.026390,63
"Piotr TQ5 (Molpro, full path)",0.763065,1.013800,11.102566,2.189494,49
Simon (CCSD(T)/aug-cc-pv5Z),0.534983,1.703231,7.183048,2.787031,10


## 8. Summary by parent molecule

Group by parent molecule family to see which systems are most sensitive.

In [16]:
def get_parent(mol_pair):
    nsi = mol_pair.split("/")[1] if "/" in mol_pair else mol_pair
    return isotopologue_to_formula(nsi)

diff_df["Parent"] = diff_df["Molecule"].apply(get_parent)

# For each parent and benchmark method, show mean absolute diff
parent_summary_rows = []
for parent in sorted(diff_df["Parent"].unique()):
    subset = diff_df[diff_df["Parent"] == parent]
    row = {"Parent": parent, "N pairs": len(subset)}
    for col in diff_cols:
        short = col.replace(" (‰)", "")
        row[f"{short} mean|‰|"] = subset[col].abs().mean()
    parent_summary_rows.append(row)

parent_summary = pd.DataFrame(parent_summary_rows).set_index("Parent")
parent_summary

,N pairs,"Amila TQ5 (Molpro, 7pt fit) mean|‰|","Piotr 7Z (Gaussian, full path) mean|‰|","Piotr F12 (Molpro/Orca, full path) mean|‰|","Piotr TQ5 (Molpro, 7pt fit) mean|‰|","Piotr TQ5 (Molpro, full path) mean|‰|",Simon (CCSD(T)/aug-cc-pv5Z) mean|‰|
Parent,,,,,,,
CO,8,0.567898,0.444437,0.336617,0.504248,0.513977,0.730155
CS,11,0.715465,0.024244,0.056806,0.020427,0.025588,NaN
Cl2,1,0.131291,0.098740,0.098399,0.064740,0.066247,NaN
H2,2,9.455083,8.119446,8.843314,9.058763,8.714671,NaN
HCl,5,5.632044,3.978100,3.801493,1.119582,1.068936,NaN
HF,2,10.060814,6.612320,5.519818,4.984590,4.939724,5.595534
N2,2,0.923903,0.211577,0.279048,0.185104,0.182336,NaN
NO,5,1.090705,0.587617,0.514950,0.897183,0.349944,NaN
O2,4,2.796256,2.307950,1.651243,1.977806,1.973774,NaN


## 9. Detailed view: contributions for a specific pair

Compare all RPFR contributions (ZPE, excited state, rotational, etc.)
across methods for a chosen isotopologue pair.

In [17]:
# --- Change this to inspect different pairs ---
MOLECULE = "13C16O/12C16O"
# -----------------------------------------------

# Amila values
amila_row = amila_df[amila_df["Molecule"] == MOLECULE]

# Experimental values
exp_row = exp_rpfr_df[exp_rpfr_df["Molecule"] == MOLECULE]

# Benchmark values (one row per method)
bench_rows_mol = bench_rpfr[bench_rpfr["Molecule"] == MOLECULE].dropna(subset=["RPFR 273.15K"])

if amila_row.empty:
    print(f"Molecule '{MOLECULE}' not found.")
else:
    records = []
    # Add Amila
    for col in AMILA_CONTRIB_COLS:
        records.append({"Contribution": col, "Source": "Amila", "Value": amila_row[col].values[0]})
    # Add Experimental
    if not exp_row.empty:
        for col in AMILA_CONTRIB_COLS:
            records.append({"Contribution": col, "Source": "Experimental", "Value": exp_row[col].values[0]})
    # Add each benchmark method
    for _, brow in bench_rows_mol.iterrows():
        method = brow["Benchmark_Method"]
        for col in AMILA_CONTRIB_COLS:
            records.append({"Contribution": col, "Source": method, "Value": brow[col]})

    detail = pd.DataFrame(records)
    detail_pivot = detail.pivot(index="Contribution", columns="Source", values="Value")
    # Reorder contributions
    detail_pivot = detail_pivot.reindex(AMILA_CONTRIB_COLS)
    print(f"=== {MOLECULE} ===")
    display(detail_pivot)

=== 13C16O/12C16O ===


Source,Amila,"Amila TQ5 (Molpro, 7pt fit)",Experimental,"Piotr 7Z (Gaussian, full path)","Piotr F12 (Molpro/Orca, full path)","Piotr TQ5 (Molpro, 7pt fit)","Piotr TQ5 (Molpro, full path)",Simon (CCSD(T)/aug-cc-pv5Z)
Contribution,,,,,,,,
Translational,1.054240,1.054240,1.054240,1.054240,1.054240,1.054240,1.054240,1.054240
ZPE (total),1.135382,1.135357,1.134980,1.134685,1.134757,1.134646,1.134639,1.134496
Excited (harmonic),1.000003,1.000003,1.000003,1.000003,1.000003,1.000003,1.000003,1.000003
Excited (anharmonic),1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
Rotational (diatomic),1.045963,1.045964,1.045964,1.045964,1.045964,1.045964,1.045964,1.045964
Rotational-vibrational,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
RPFR 273.15K,1.251984,1.251957,1.251542,1.251217,1.251296,1.251173,1.251166,1.251008


## 10. All pairs sorted by largest benchmark spread

Which isotopologue pairs show the most sensitivity to the choice of ωe method?

In [18]:
# For each pair, compute the range (max - min) of RPFR across benchmark methods
spread = rpfr_compare.copy()
spread["Bench_max"] = spread[bench_method_cols].max(axis=1)
spread["Bench_min"] = spread[bench_method_cols].min(axis=1)
spread["Bench_range"] = spread["Bench_max"] - spread["Bench_min"]
spread["Bench_range_permil"] = (spread["Bench_range"] / spread["Amila"]) * 1000.0

spread_sorted = spread.sort_values("Bench_range_permil", ascending=False)
display(
    spread_sorted[["Molecule", "Amila", "Bench_min", "Bench_max", "Bench_range", "Bench_range_permil"]]
    .rename(columns={"Bench_range_permil": "Range (‰)"})
    .head(20)
)

,Molecule,Amila,Bench_min,Bench_max,Bench_range,Range (‰)
8,3H37Cl/1H35Cl,78.354920,77.958433,78.482705,0.524272,6.690987
5,3H35Cl/1H35Cl,71.533400,71.172219,71.649351,0.477132,6.670062
3,3H19F/1H19F,208.323600,208.165294,209.414327,1.249033,5.995637
7,2H37Cl/1H35Cl,19.782380,19.714345,19.806662,0.092317,4.666640
4,2H35Cl/1H35Cl,18.078870,18.017213,18.101268,0.084055,4.649378
2,2H19F/1H19F,38.205310,38.196595,38.356600,0.160005,4.188033
16,15N18O/14N16O,1.582121,1.576790,1.581995,0.005205,3.289636
15,15N17O/14N16O,1.380695,1.377331,1.380586,0.003255,2.357277
51,14C18O/12C16O,2.055315,2.050524,2.055201,0.004677,2.275601
13,14N18O/14N16O,1.326420,1.323778,1.326351,0.002572,1.939287
